# 01 · Individual Engagement
Drill into a single specialist's customer meetings, conversation themes, and use case association.
Adjust the filters in the first cell to change the specialist or date range.

In [ ]:
import datetime, json
import matplotlib.pyplot as plt, matplotlib.patches as mpatches, matplotlib.gridspec as gs
import pandas as pd
from IPython.display import display, HTML
plt.switch_backend("agg")

SF_BLUE="#29B5E8"; SF_TEAL="#00A9CE"; SF_GREEN="#36B37E"
SF_ORANGE="#FF8B00"; SF_DARK="#0A2859"; SF_GRAY="#D0D0D0"; BG="#F9FAFB"

def clean(ax):
    ax.set_facecolor(BG)
    for s in ax.spines.values(): s.set_visible(False)

def fmt_eacv(v):
    if not v or (isinstance(v,float) and str(v)=="nan"): return "$0"
    v=float(v)
    return f"${v/1e6:.1f}M" if v>=1e6 else f"${v/1e3:.0f}K" if v>=1e3 else f"${v:.0f}"

def html_table(df, row_color_fn=None):
    if hasattr(df, 'to_pandas'): df = df.to_pandas()
    html = "<table style='border-collapse:collapse;width:100%;font-size:13px;font-family:sans-serif'>"
    html += "<tr style='background:#0A2859;color:white'>" + "".join("<th style='padding:8px 12px;text-align:left'>" + str(col) + "</th>" for col in df.columns) + "</tr>"
    for i, (_, r) in enumerate(df.iterrows()):
        bg = row_color_fn(r) if row_color_fn else ("#f8f9fa" if i%2==0 else "white")
        html += "<tr style='background:" + bg + "'>" + "".join("<td style='padding:7px 12px'>" + str(v if v is not None else '') + "</td>" for v in r) + "</tr>"
    html += "</table>"
    display(HTML(html))

TEAM = {
    "Jim Lebonitte": "005VI00000Z0y6bYAB",
    "Michael Hamilton": "005VI00000ExC3VYAV",
    "Patrick Sheehan": "0053r00000BblkZAAR",
    "Phani Alapaty": "005VI00000HajknYAB",
    "Steve Mitchener": "0050Z000009IrKRQA0",
    "Zaki Bajwa": "005VI00000XibQfYAJ"
}
TEAM_IDS   = list(TEAM.values())
team_ids_str = "', '".join(TEAM_IDS)
ACT_TABLE  = "TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES"

fy_start   = datetime.date(2026, 2, 1)
q2_start   = datetime.date(2026, 5, 1)
today      = datetime.date.today()
fy_start_str = str(fy_start)
q2_start_str = str(q2_start)
today_str    = str(today)
print("Setup complete ✓")

# ─── FILTERS ─────────────────────────────────────────────
selected_member  = "Jim Lebonitte"   # change to any team member
period_start     = datetime.date(2026, 4, 1)
period_end       = today
# ─────────────────────────────────────────────────────────
selected_id      = TEAM[selected_member]
ps, pe = str(period_start), str(period_end)
print(f"Specialist: {selected_member}  |  {ps} → {pe}")


## Activity Volume by Week

In [ ]:
%%sql -r weekly_vol
SELECT
    DATE_TRUNC('week', MEETING_DATE)::DATE AS week,
    COUNT(*)                               AS meetings,
    COUNT(DISTINCT SF_ACCOUNT_ID)          AS unique_accounts,
    SUM(IFF(SUMMARY IS NOT NULL, 1, 0))    AS with_gong_summary
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES
WHERE MEETING_SE_NAME ILIKE '%{{selected_member}}%'
  AND MEETING_DATE BETWEEN '{{ps}}' AND '{{pe}}'
GROUP BY 1 ORDER BY 1

In [ ]:
weekly_vol = weekly_vol.to_pandas() if hasattr(weekly_vol, "to_pandas") else weekly_vol
df = weekly_vol.to_pandas() if hasattr(weekly_vol,'to_pandas') else weekly_vol
fig, ax = plt.subplots(figsize=(13,3.5)); clean(ax)
weeks = df["WEEK"].astype(str)
ax.bar(weeks, df["MEETINGS"], color=SF_BLUE, alpha=0.85, label="Meetings")
ax.bar(weeks, df["WITH_GONG_SUMMARY"], color=SF_TEAL, alpha=0.85, label="With Gong Summary")
ax2 = ax.twinx()
ax2.plot(weeks, df["UNIQUE_ACCOUNTS"], color=SF_ORANGE, marker="o", linewidth=2, label="Unique Accounts")
ax.set_title(f"Weekly Customer Activity — {selected_member}", fontsize=13, color=SF_DARK, pad=10)
ax.set_ylabel("Meetings", color=SF_DARK); ax2.set_ylabel("Unique Accounts", color=SF_ORANGE)
lines = [mpatches.Patch(color=SF_BLUE, label="Meetings"),
         mpatches.Patch(color=SF_TEAL, label="With Gong Summary"),
         mpatches.Patch(color=SF_ORANGE, label="Unique Accounts")]
ax.legend(handles=lines, loc="upper left", frameon=False)
plt.xticks(rotation=40, ha="right"); plt.tight_layout(); plt.show()

## Top Accounts — Engagement & Use Case Status

In [ ]:
%%sql -r top_accts
SELECT
    COALESCE(CUSTOMER,'(unknown)')        AS customer,
    MAX(SF_ACCOUNT_ID)                    AS sf_account_id,
    COUNT(*)                              AS meetings,
    MAX(MEETING_DATE)                     AS last_touch,
    DATEDIFF('day', MAX(MEETING_DATE), CURRENT_DATE()) AS days_since,
    MAX(USE_CASE_TAGGED_IN_SF)            AS uc_status,
    MAX(USE_CASE_NAME)                    AS use_case,
    MAX(OPPORTUNITY_NAME)                 AS opportunity
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES
WHERE MEETING_SE_NAME ILIKE '%{{selected_member}}%'
  AND MEETING_DATE BETWEEN '{{ps}}' AND '{{pe}}'
  AND CUSTOMER IS NOT NULL AND CUSTOMER != ''
GROUP BY 1 ORDER BY 3 DESC LIMIT 25

In [ ]:
top_accts = top_accts.to_pandas() if hasattr(top_accts, "to_pandas") else top_accts
def row_color(r):
    s = str(r.get("UC_STATUS",""))
    d = int(r.get("DAYS_SINCE") or 0)
    if s == "Yes": return "#d4edda"
    if d > 30: return "#fdecea"
    if s == "No": return "#fff3cd"
    return "white"
html_table(top_accts, row_color_fn=row_color)
print("Green=Use case tagged  |  Yellow=No use case  |  Red=No touch >30 days")

## Conversation Themes — AI Analysis of Gong Summaries

In [ ]:
%%sql -r gong_summaries
SELECT MEETING_DATE, CUSTOMER, MEETING_TITLE, SUMMARY, NEXT_STEPS
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES
WHERE MEETING_SE_NAME ILIKE '%{{selected_member}}%'
  AND MEETING_DATE BETWEEN '{{ps}}' AND '{{pe}}'
  AND SUMMARY IS NOT NULL AND TRIM(SUMMARY) != ''
ORDER BY MEETING_DATE DESC LIMIT 40

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
gong_summaries = gong_summaries.to_pandas() if hasattr(gong_summaries, "to_pandas") else gong_summaries
texts = "\n---\n".join(
    f"[{r['MEETING_DATE']} - {r['CUSTOMER']}] {r['SUMMARY']}"
    for _, r in gong_summaries.iterrows() if r['SUMMARY']
)
if texts.strip():
    prompt = (
        f"You are analyzing Gong call summaries for {selected_member}, a Snowflake specialist.\n"
        "Identify the top 5 customer themes and business priorities you see across these conversations. "
        "For each theme include: theme name, frequency signal, and a 1-sentence insight.\n"
        "Also note any competitor mentions or product gaps.\n\nSummaries:\n" + texts[:5000]
    )
    result = session.sql("SELECT SNOWFLAKE.CORTEX.COMPLETE('mistral-7b', ?)", [prompt]).collect()[0][0]
    print(f"=== Key Themes for {selected_member} ({ps} → {pe}) ===\n")
    print(result)
else:
    print("No Gong summaries in this period — run the weekly report with --gsheet to populate.")

## Salesforce Logging & Use Case Coverage

In [ ]:
%%sql -r logging_stats
SELECT
    COUNT(*)                                                          AS total_meetings,
    SUM(IFF(SF_ACTIVITY_ID IS NOT NULL, 1, 0))                        AS setsail_matched,
    SUM(IFF(SF_ACTIVITY_ID IS NULL, 1, 0))                            AS backlog,
    ROUND(100.0*SUM(IFF(SF_ACTIVITY_ID IS NOT NULL,1,0))/COUNT(*),1) AS setsail_match_pct,
    SUM(IFF(USE_CASE_TAGGED_IN_SF='Yes',1,0))                         AS with_uc,
    SUM(IFF(USE_CASE_TAGGED_IN_SF='No' AND SF_ACCOUNT_ID IS NOT NULL,1,0)) AS uc_gap,
    ROUND(100.0*SUM(IFF(USE_CASE_TAGGED_IN_SF='Yes',1,0))/NULLIF(SUM(IFF(SF_ACCOUNT_ID IS NOT NULL,1,0)),0),1) AS uc_pct
FROM TEMP.JLEBONITTE_EDA_ACTIVITY_TRACKING.SE_WEEKLY_ACTIVITIES
WHERE MEETING_SE_NAME ILIKE '%{{selected_member}}%'
  AND MEETING_DATE BETWEEN '{{ps}}' AND '{{pe}}'

In [ ]:
logging_stats = logging_stats.to_pandas() if hasattr(logging_stats, "to_pandas") else logging_stats
r = logging_stats.iloc[0]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax in axes: clean(ax)
metrics = [
    (float(r.get("SETSAIL_MATCH_PCT",0) or 0), "Setsail Matched", SF_BLUE),
    (float(r.get("UC_PCT",0) or 0),     "Accounts with Use Case", SF_GREEN),
    (100 - float(r.get("SETSAIL_MATCH_PCT",0) or 0), "Not Matched by Setsail", SF_ORANGE),
]
for ax, (val, title, color) in zip(axes, metrics):
    ax.pie([val, 100-val], colors=[color, SF_GRAY], startangle=90, wedgeprops={"linewidth":0})
    ax.text(0, 0, f"{val:.0f}%", ha="center", va="center", fontsize=20, fontweight="bold", color=SF_DARK)
    ax.set_title(title, color=SF_DARK, fontsize=11)
plt.suptitle(f"{selected_member} · {ps} → {pe}", color=SF_DARK, fontsize=12)
plt.tight_layout(); plt.show()
print(f"Total: {r['TOTAL_MEETINGS']} | Logged: {r['SETSAIL_MATCHED']} | Setsail unmatched: {r['BACKLOG']} | UC Gap: {r['UC_GAP']} accounts need use case")